In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root

In [2]:
import rasterio
from pathlib import Path

from spm.config.config import ModelConfig
from spm.models.yolo import YOLOModel
from spm.utils.profiling import size_it, time_it
from spm.visualization.overlays import visualize

from sahi.postprocess.combine import GreedyNMMPostprocess
from sahi.postprocess.backends import set_postprocess_backend

from utils.adapters import PredictionAdapter, SAHIPrediction

## Run Full Inference Pipeline

In [3]:
model_path = "../runs/segment/yolo-seg-whu/weights/best.pt"
tiff_path = Path("../whole_cropped_2500m.tif")

In [4]:
# Inference configuration
tile_size = 1500
batch_size = 4
overlap = 0.2
device = "cuda"  # or "cpu"

In [5]:
config = ModelConfig(
    model_path=model_path,
    tile_size=tile_size,
    batch_size=batch_size,
    overlap=overlap,
    device=device,
    )

In [6]:
model = YOLOModel(config)

In [7]:
prediction, unmerged_prediction = model(
                                    tiff_path,
                                    vector_format="gpkg",
                                    show_viz=False,
                                    merge_only_border=False,
                                    get_seg_from_binary_mask=True
                                    )

[2026-06-29 02:07:40.499042] INFO - Performing prediction on image: ../whole_cropped_2500m.tif (width: 33333, height: 33333) (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:163:predict())
[2026-06-29 02:09:05.833014] INFO - Total tiles processed: 784 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:258:predict())
[2026-06-29 02:09:05.834009] INFO - Indexing polygons for merging (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:20:index())
[2026-06-29 02:09:05.843014] INFO - Execution time for index(): 00:00:00.008987 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-29 02:09:05.982407] INFO - Starting merge process with 18296 polygons. (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/merging/spm.py:131:merge())
[2026-06-29 02:09:08.921354] INFO - Total merged polygons: 5381 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/me

In [8]:
# Adapt the unmerged prediction to support both SAHI and SMM formats
prediction_adapter = PredictionAdapter(unmerged_prediction)
unmerged_sahi_predictions = prediction_adapter.sahi
unmerged_smm_predictions = prediction_adapter.smm

## Merge Predictions with SAHI(GreedyNMM)

In [9]:
postprocess = GreedyNMMPostprocess(
        match_threshold=0.1,
        match_metric="IOS",
        class_agnostic=False,
    )

In [10]:
# Set SAHI backend to numpy to process predictions on CPU
set_postprocess_backend("numpy")

In [11]:
@size_it
@time_it
def _postprocess(sahi_predictions):
    merged_sahi_predictions = postprocess(sahi_predictions.predictions)
    return merged_sahi_predictions

In [12]:
# Postprocess the unmerged SAHI predictions to merge overlapping polygons
merged_sahi_predictions = _postprocess(unmerged_sahi_predictions)

[2026-06-29 02:09:24.731175] INFO - Execution time for _postprocess(): 00:00:12.428608 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:18:wrapper())
[2026-06-29 02:09:24.785470] INFO - Peak memory usage for _postprocess(): 5157.43 MB (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/utils/profiling.py:33:wrapper())


In [13]:
# Convert merged SAHI predictions back to SPM format
sahi_prediction = SAHIPrediction()
sahi_prediction.predictions = merged_sahi_predictions
sahi_merged_predictions_spm = PredictionAdapter(sahi_prediction).spm

In [ ]:
# Visualize the merged predictions on the original TIFF image
viz_dir = Path("predictions/viz")
viz_dir.mkdir(parents=True, exist_ok=True)
output_path = viz_dir / f"{tiff_path.stem}_sahi_gnmm_prediction.png"
with rasterio.open(tiff_path) as src:
    visualize(src, sahi_merged_predictions_spm, output_path)